# Step 8a — export a self-contained bundle for the tabular foundation models

The bundle holds everything `step8_foundation_models.ipynb` needs, so the GPU machine needs neither NHANES data nor this pipeline: the re-encoded 86 variables of Dang et al. (2024) (step 6), the 0/1 label, the PHQ-9 total (step 7) and the paper's 10-fold split (the same StratifiedKFold(10, shuffle, seed 42) as steps 6 and 7, so predictions can be paired fold by fold).

**Inputs** (with `RUN`): `PAPER1_REPRO`, `PAPER1_CSV` as in step 6; run step 6 first.
**Output:** `$NHANES_WORK/modeling/step8_bundle/` `data.csv.gz` (SEQN, fold, depression, phq9, then the 86 features) and `columns.json` (`{"categorical": [...], "continuous": [...]}`)

`RUN = True` writes the bundle (seconds); `RUN = False` only shows the saved one.

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "nhanes" / "paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("run this notebook from inside the repository")
sys.path.insert(0, str(ROOT))
from nhanes.paths import RESULTS  # noqa: E402

RUN = False  # True: write the bundle
PAPER1_REPRO = os.environ.get("PAPER1_REPRO")  # folder with build_dataset.py of the paper1 reproduction
PAPER1_CSV = os.environ.get("PAPER1_CSV")      # its FullData_rm_dl_bt.csv
RUN = RUN or os.environ.get("NHANES_RUN") == "1"
OUT = RESULTS / "step8_bundle"

## Run

In [2]:
if RUN:
    from sklearn.model_selection import StratifiedKFold

    from nhanes.paper1 import paper1_table, phq9_totals

    feats, dic = paper1_table(RESULTS / "step6_paper1_recoded" / "data" / "FullData_recoded.csv")
    y = pd.read_csv(PAPER1_CSV, usecols=["depression"]).depression.eq("Depressed").astype(int).to_numpy()
    fold = np.empty(len(y), dtype=int)
    for k, (_, te) in enumerate(StratifiedKFold(10, shuffle=True, random_state=42).split(feats, y)):
        fold[te] = k
    phq = feats.SEQN.map(phq9_totals(PAPER1_REPRO))
    assert phq.notna().all() and ((phq >= 10).astype(int).to_numpy() == y).all()
    cols = [c for c in feats if c != "SEQN"]
    out = pd.concat([pd.DataFrame({"SEQN": feats.SEQN, "fold": fold, "depression": y, "phq9": phq.astype(int)}),
                     feats[cols]], axis=1)
    OUT.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT / "data.csv.gz", index=False)
    (OUT / "columns.json").write_text(json.dumps(
        {"categorical": [c for c in cols if dic[c] == "categorical"],
         "continuous": [c for c in cols if dic[c] != "categorical"]}, indent=1))
    print(f"{len(out)} people, {len(cols)} features, depression {y.mean():.3f}")

## The bundle

In [3]:
data = pd.read_csv(OUT / "data.csv.gz")
cols = json.loads((OUT / "columns.json").read_text())
print(f"{len(data):,} people, {len(cols['categorical'])} categorical + {len(cols['continuous'])} continuous features, "
      f"depression {data.depression.mean():.3f}")
display(data.groupby("fold").agg(n=("SEQN", "size"), depression=("depression", "mean"), phq9_mean=("phq9", "mean")).round(3))

36,259 people, 54 categorical + 32 continuous features, depression 0.087


,n,depression,phq9_mean
fold,,,
0,3626,0.087,3.222
1,3626,0.087,3.227
2,3626,0.087,3.200
3,3626,0.087,3.190
4,3626,0.087,3.190
5,3626,0.087,3.245
6,3626,0.087,3.171
7,3626,0.087,3.162
8,3626,0.087,3.197
